In [694]:
 # # Import modules
import pandas as pd

In [695]:
# # set filter criteria
Filter_ttv_flag = 1
Filter_EarthRadius_lower = 3.0
Filter_EarthRadius_upper = 8.5
Filter_Vmag = 13.0

In [696]:
# ratio of radii . Earth to Jupiter. Carole Haswell Book. used to convert Jupiter Radii to Earth equivalents
radius_earth_m = 6.37e6
radius_jupiter_m = 7.15e7
ratio_earth_to_jupiter = radius_earth_m/radius_jupiter_m

In [697]:
# # Define planet classification ranges
df_NASA_ranges = pd.DataFrame({"lower":[0,0.5,1.0,1.75,3.5,6.0,14.3],\
                               "upper": [0.5,1.0,1.75,3.5,6.0,14.3,9999],\
                               "classification": ["1.Radius below lower bound","2.Rocky planet","3.Super Earth","4.Sub Neptune","5.Neptune","6.Jupiters","7.Radius above upper bound"] \
                              })
intervals = pd.IntervalIndex.from_arrays(df_NASA_ranges["lower"],df_NASA_ranges["upper"],closed="both")

In [698]:
# # Function to set planet classifications
def classify_exoplanet(radius):
    match = df_NASA_ranges.loc[intervals.contains(radius),"classification"]
    return match.iloc[0] if not match.empty else None

In [699]:
# Define function to clean exoplanet names
def clean_name(s):
    return (
        s.str.lower()
         .str.replace(" ", "", regex=False)
         .str.replace("-", "", regex=False)
         .str.replace("_", "", regex=False)
         .str.replace("[", "", regex=False)
         .str.replace("]", "", regex=False)
         .str.replace("'", "", regex=False)
    )

In [700]:
# # Load raw data files
# use low_memory = False to avoid warning message about  mixed data types in columns 4 and 5
df_NASA_data = pd.read_csv("/Users/danielbhuglah/Downloads/PS_2026.01.05_06.10.12.CSV", skiprows=292, low_memory = False)
df_EU_data = pd.read_csv("/Users/danielbhuglah/Downloads/exoplanet.eu_catalog_02-01-26_16_43_00.CSV")

In [701]:
###################################################
# # NASA Processing
###################################################
# NASA - only select exoplanet data where 
# i) default flag is 1 and ii) the planet has been confirmed. iii) the radius is within defined range 
# iv) TTV flag is 1 v) the Vmag is less than or equal to the defined value
df_NASA_data_filtered = df_NASA_data[(df_NASA_data["default_flag"] == 1) &
                            (df_NASA_data["soltype"] == "Published Confirmed") &
                            (df_NASA_data["ttv_flag"] == Filter_ttv_flag) &
                            (df_NASA_data["pl_rade"] >= Filter_EarthRadius_lower) &
                            (df_NASA_data["pl_rade"] <= Filter_EarthRadius_upper) &
                            (df_NASA_data["sy_vmag"] <= Filter_Vmag)
                            ]


In [702]:
# # Set exoplanet classification. Copy the filtered DF to ensure no issues with updating when you do the exoplant classification. 
df_NASA_data_filtered_V2 = df_NASA_data_filtered.copy()
df_NASA_data_filtered_V2["Exoplanet_class"] = df_NASA_data_filtered_V2["pl_rade"].apply(classify_exoplanet)

In [703]:
# # Download the filtered NASA file
df_NASA_data_filtered_V2.to_excel("/Users/danielbhuglah/Downloads/NASA_data_filtered_V2.xlsx",index=False)

In [704]:
###################################################
# # EU Processing
###################################################

# Raw data file loaded earlier in code
# Covert the eu planet radius (in Jupiter eqivalents) into earth radius equivalents.
#used to do classifications and normalise with NASA archives
df_EU_data["radius_earth"]= df_EU_data["radius"]/ ratio_earth_to_jupiter

In [705]:
# use the earth radius equivalents to assign exoplanet classificaitons
df_EU_data["Exoplanet_class"] = df_EU_data["radius_earth"].apply(classify_exoplanet)

In [706]:
# Look to join EU and NASA files. 
# TTV_flag is only held on the NASA file. 
# Also, if a planet is missing radius or v_mag in the EU file, try to pull from the NASA file.

# clean names by removing blanks, dashes and underscores. This should help joining the two files.
df_EU_data["clean_name"] = clean_name(df_EU_data["name"])
df_NASA_data["clean_name"] = clean_name(df_NASA_data["pl_name"])

# select default records only from the NASA file
df_NASA_data_defaults = df_NASA_data[df_NASA_data["default_flag"] == 1].copy()

In [714]:
df_NASA_data_defaults.to_excel("/Users/danielbhuglah/Downloads/df_NASA_defaults_DAN.xlsx",index=False)

In [707]:
# explode the alternate name list from eu data and create one row per exploded name
# e.g 54 Psc b has alternate names '54 Psc Ab' and 'HD 3651 b'
# alt_list is created and contains a list ['54 Psc Ab','HD 3651 b']
df_EU_data["alt_list"]= df_EU_data["alternate_names"].str.split(",")

# df_EU_data.to_excel("/Users/danielbhuglah/Downloads/df_EU_data_alt_list_DAN.xlsx",index=False)

In [708]:
# create one row for each alternate name value
df_EU_data_exploded = df_EU_data.explode("alt_list")
# clean alternate names by removing blanks, dashes and underscores. This should help joining the two (EU and NASA) files.
df_EU_data_exploded["clean_alt"] = clean_name(df_EU_data_exploded["alt_list"])
# If the record did not have any alternate names then force the orignal record into clean_alt. Makes joining to NASA data easier
#mask = df_EU_data_exploded["clean_alt"].isna()
#df_EU_data_exploded.loc[mask, "clean_alt"] = df_EU_data_exploded.loc[mask, "clean_name"]

In [709]:
df_EU_data_exploded.to_excel("/Users/danielbhuglah/Downloads/df_EU_data_exploded_DAN.xlsx",index=False)

In [710]:
# setup columns to be pulled in from NASA data
NASA_cols = ["clean_name","ttv_flag"]
# merge the EU data with NASA equivalents - left merge
#df_merged_main = df_EU_data.merge(df_NASA_data,on="clean_name",how="left",suffixes=("","_nasa"))
#df_merged_main = df_EU_data_exploded.merge(df_NASA_data_defaults,left_on="clean_alt",right_on="clean_name",how="left",suffixes=("","_nasa"))

# merge the unexploded EU data with the NASA data. this uses the original planet names (excluding blanks, underscores etc.)
# use an inner join so df_merged_EU_data only included records where data exists in both the EU and NASA files.
df_merged_EU_data = df_EU_data.merge(df_NASA_data_defaults,on="clean_name",how="inner",suffixes=("","_nasa"))

# merge the exploded EU data, using the planet's alternate names with NASA's clean name. If the alternate name does not exist then the record is dropped
df_merged_EU_data_exploded = df_EU_data_exploded.merge(df_NASA_data_defaults,left_on="clean_alt",right_on="clean_name",how="inner",suffixes=("","_nasa"))

# Combine the matched records from the unexploded and exploded dataframes
df_merged_main = pd.concat([df_merged_EU_data,df_merged_EU_data_exploded],ignore_index=True)

In [711]:
# get a list of planets in the EU file, that have NOT matched to the NASA file.
df_EU_planets_not_matched_to_NASA = df_EU_data.merge(df_merged_main[["name"]],on="name",how="left",indicator=True)

df_EU_planets_not_matched_to_NASA = df_EU_planets_not_matched_to_NASA[df_EU_planets_not_matched_to_NASA["_merge"] == "left_only"].drop(columns="_merge")

In [712]:
df_EU_planets_not_matched_to_NASA.to_excel("/Users/danielbhuglah/Downloads/EU_planets_not_in_NASA.xlsx",index=False)

In [713]:
df_merged_main.to_excel("/Users/danielbhuglah/Downloads/merged_data_DAN.xlsx",index=False)

In [ ]:
# In the merged data frame, drop duplicate rows from EU planets that have not matched to the NASA file
#df_merged_main["pl_name"].replace("",pd.NA, inplace=True)
#df_merged_main.replace({"pl_name": {"": pd.NA}}, inplace=True)

#mask_blank = df_merged_main["pl_name"].isna()

#df_merged_main.loc[mask_blank] = df_merged_main.loc[mask_blank].drop_duplicates(subset="name")

# after dropping duplicates remove blanks rows
#df_merged_main = df_merged_main.dropna(how="all")


In [ ]:
# if the exoplanet radius in the EU file is not blank put this into a new column else take the radius from the NASA value pl_rade.
# Then reclassify the exoplanet based on its size
# same approach for the Vmag.
df_merged_main["radius_earth"] = df_merged_main["radius_earth"].replace("", pd.NA)
df_merged_main["merged_radius_earth"] = df_merged_main["radius_earth"].fillna(df_merged_main["pl_rade"])
# use the earth radius equivalents to assign exoplanet classificaitons
df_merged_main["merged_Exoplanet_class"] = df_merged_main["merged_radius_earth"].apply(classify_exoplanet)

df_merged_main["mag_v"] = df_merged_main["mag_v"].replace("", pd.NA)
df_merged_main["merged_vmag"] = df_merged_main["mag_v"].fillna(df_merged_main["sy_vmag"])

In [ ]:
df_merged_main.to_excel("/Users/danielbhuglah/Downloads/merged_data_DAN.xlsx",index=False)

In [ ]:
# # EU Processing
# EU - only select exoplanet data where 
# i) default flag is 1 and ii) the planet has been confirmed. iii) the radius is within defined range 
# iv) TTV flag is 1 v) the Vmag is less than or equal to the defined value
df_EU_data_merged_filtered = df_merged_main[(df_merged_main["planet_status"] == "Confirmed") &
                            (df_merged_main["ttv_flag"] == Filter_ttv_flag) &
                            (df_merged_main["merged_radius_earth"] >= Filter_EarthRadius_lower) &
                            (df_merged_main["merged_radius_earth"] <= Filter_EarthRadius_upper) &
                            (df_merged_main["merged_vmag"] <= Filter_Vmag)
                            ]


In [ ]:
df_EU_data_merged_filtered.to_excel("/Users/danielbhuglah/Downloads/EU_data_merged_filtered.xlsx",index=False)